In [ ]:
from langgraph.graph import StateGraph, START, END, MessagesState
from langgraph.checkpoint.memory import InMemorySaver

FIRST_STAGE = "first_stage"
SECOND_STAGE = "second_stage"
FINAL_STAGE = "final_stage"
process_status = {"is_interruption":True}

In [ ]:
class MsgState(MessagesState):
    user_request:str
    response:str

In [ ]:
def process_first_stage(state: MsgState):
    print("\nFirst Stage Process Begins...")
    state["user_request"] = state["user_request"] + "- Stage One"
    state["response"] = f"{state["user_request"]} - Clearance from First stage process"
    print(state)
    return state

def process_second_stage(state: MsgState):
    print("\nSecond Stage Process Begins...")
    state["user_request"] = state["user_request"] + "- Stage Two"
    if process_status["is_interruption"]:
        process_status["is_interruption"] = False
        print("\nProcess Interrupted")
        raise ValueError("interrupted due to raw material mix")
    state["response"] = f"{state["user_request"]} - Clearance from Second stage process"
    print(state)
    return state

def process_final_stage(state:MsgState):
    print("\nFinal Stage Process Begins...")
    state["user_request"] = state["user_request"] + "- Process Completed"
    state["response"] = f"{state["user_request"]} - Final Clearance -- Product is ready"
    print(state)
    return state

In [ ]:
wf_graph = StateGraph(MsgState)
wf_graph.add_node(FIRST_STAGE, process_first_stage)
wf_graph.add_node(SECOND_STAGE, process_second_stage)
wf_graph.add_node(FINAL_STAGE, process_final_stage)

wf_graph.add_edge(START, FIRST_STAGE)
wf_graph.add_edge(FIRST_STAGE, SECOND_STAGE)
wf_graph.add_edge(SECOND_STAGE, FINAL_STAGE)
wf_graph.add_edge(FINAL_STAGE, END)

in_memory_checkpoint = InMemorySaver()
graph = wf_graph.compile(in_memory_checkpoint)

### Checkpointer with Configurable - Thread Id - User-1

In [ ]:
configs = {"configurable":{"thread_id":"user-1"}}
#response = graph.invoke({"messages":[{"user_request":"Refine Oil"}]}, configs)
#response = graph.invoke({"messages":[{"role":"user", "content":"Refine Oil"}]}, configs)
response = graph.invoke({"user_request":"Refine Oil"}, config=configs)

print(response)

### Checkpointer with Configurable - Thread Id - User-2

In [ ]:
configs1 = {"configurable":{"thread_id":"user-2"}}
response = graph.invoke({"user_request":"Refine Palm Oil"}, config=configs1)
print(response)

In [25]:
#print(in_memory_checkpoint.get(config=configs1))
in_memory_checkpoint.get_tuple(config=configs1)

CheckpointTuple(config={'configurable': {'thread_id': 'user-2', 'checkpoint_ns': '', 'checkpoint_id': '1f15dae9-a9ee-69b0-800b-37fad2295ab6'}}, checkpoint={'v': 4, 'ts': '2026-06-01T11:39:52.499442+00:00', 'id': '1f15dae9-a9ee-69b0-800b-37fad2295ab6', 'channel_versions': {'__start__': '00000000000000000000000000000010.0.748722210753653', 'user_request': '00000000000000000000000000000013.0.10688066742987057', 'branch:to:first_stage': '00000000000000000000000000000011.0.0390308846935189', 'messages': '00000000000000000000000000000013.0.10688066742987057', 'response': '00000000000000000000000000000013.0.10688066742987057', 'branch:to:second_stage': '00000000000000000000000000000012.0.15320386667937502', 'branch:to:final_stage': '00000000000000000000000000000013.0.10688066742987057'}, 'versions_seen': {'__input__': {}, '__start__': {'__start__': '00000000000000000000000000000009.0.513129497544579'}, 'first_stage': {'branch:to:first_stage': '00000000000000000000000000000010.0.74872221075365